# 7.3 Naive Bayes Classifier based on Bernoulli distributions

In the previous notebook we built the discriminant functions of a Bayesian classifier, $d_k(\mathbf{x}) = \ln p(\mathbf{x}|C_k) + \ln P(C_k)$ (the MAP discriminant), where the feature vectors $\mathbf{x}$ of each class followed a Gaussian distribution. In this notebook we are going to explore the case in which $\mathbf{x}$ follows a **set of Bernoulli distributions**, where the features in $\mathbf{x}$ are either $0$ or $1$ (Bernoulli trial).

This time we are assuming the **naive** Bayes classifier model (see notebook 7.2), so features are considered **conditionally independent** given the class:

$$p(x_i, x_j|C_k) = p(x_i|C_k)\,p(x_j|C_k) \quad \forall i \neq j$$

In this notebook we will learn how a naive Bayes classifier can be trained (section 7.3.1) and tested (section 7.3.2) using features following Bernoulli distributions.

## Problem context - Digit recognition

Digit recognition systems aim to recognize the digits appearing in different sources like emails, bank cheques, papers, images, etc. They have many real-world applications as diverse as online handwriting recognition on papers, computers, or tablets, the recognition of license plate numbers in vehicles, the processing of bank cheques' amounts, etc. This task is a particular case of **Optical Character Recognition** (OCR), which converts images of printed or handwritten text into machine-encoded text.

For completing the plate detector and recognition system in which we worked on in previous chapters, we are going to learn how to construct a digit recognition system using a naive Bayes classifier based on the Bernoulli distribution. This system could be generalized to recognize any possible character appearing in the place.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/plateRecognition.png" width="400"/>
    <figcaption>Fig. 1. License plate recognition: the plate is segmented and binarized, and then each character is recognized.</figcaption>
</center>$\\[5pt]$

For this problem we need a set of digit images that will be used as training dataset for our classifier. For this application we would need to classify both digits and letters, however, to simplify this, we are going to use only digit images. In this way, the problem is reduced to 10 possible classes. You can find the provided images in `./images/train_binary/imagen{0-9}_{1-500}.png`, having 500 images for each digit.

*Note that the provided images contain handwritten digits instead of plate ones, but the training and working principles are the same.*

In [ ]:
# In case of using Google Colab, mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # If your notebook and images are in your Drive, change directory accordingly:
    # %cd '/content/drive/MyDrive/Colab Notebooks/uma_cv_2027/Chapter 07. Object Recognition'
except ImportError:
    pass

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (16.0, 8.0)

images_path = './images/'

### 7.3.1 Training a naive Bayes classifier (Bernoulli distribution)

We will follow the same procedure as in the previous notebook, but adapted to the Bernoulli distribution. First, we need to remember how the discriminant function of a Bayesian classifier is created:$\\[10pt]$

$$
\begin{aligned}
d_k(\mathbf{x}) &= P(C_k|\mathbf{x}) = \frac{p(\mathbf{x}|C_k)P(C_k)}{p(\mathbf{x})}\\
&\Big\Downarrow \ \ p(\mathbf{x}) \text{ does not depend on } C_k\\
d_k(\mathbf{x}) &= p(\mathbf{x}|C_k)P(C_k)\\
&\Big\Downarrow \ \ \ln \text{ is monotonic}\\
d_k(\mathbf{x}) &= \ln p(\mathbf{x}|C_k) + \ln P(C_k)\\
&\Big\Downarrow \ \ \text{if } P(C_i) = P(C_j)\; \forall i,j\\
d_k(\mathbf{x}) &= \ln p(\mathbf{x}|C_k)
\end{aligned}
$$

In the context of our digit recognition problem, we are going to deal with images with $28\times28$ pixels, where each pixel is a feature that can take the value $0$ (background) or $1$ (stroke of the digit). These images are rearranged as vectors with $N = 28\times28 = 784$ features:

$$\mathbf{x} = [x_1 \; x_2 \; \ldots \; x_f \; \ldots \; x_N]^T$$

This way, each image could be interpreted as a point in a space with 784 dimensions. Fascinating!

We have $M=10$ classes $C_k$, $k=0,\ldots,9$, one for each digit. Considering the Bernoulli distribution, the probability of the value of a pixel $x_f$ given a class $C_k$ is modeled as:

$$p(x_f|C_k) = p_{k,f}^{\,x_f}\,(1 - p_{k,f})^{1 - x_f}$$

Being:

- $p_{k,f}$ the probability that $x_f = 1$ if $\mathbf{x}$ belongs to $C_k$, and
- $1-p_{k,f}$ the probability that $x_f = 0$ if $\mathbf{x}$ belongs to $C_k$.

Again, we are assuming that the features are conditionally independent given the class (we are building a **naive Bayes classifier**), so the probability $p(\mathbf{x}|C_k)$ can be expressed as the product of each individual probability $p(x_f|C_k)$, that is:$\\[5pt]$

$$p(\mathbf{x}|C_k) = \prod_{f=1}^N p(x_f|C_k) = \prod_{f=1}^N p_{k,f}^{\,x_f}\,(1 - p_{k,f})^{1 - x_f}$$

Applying logarithms:$\\[7pt]$

$$
\begin{aligned}
\ln p(\mathbf{x}|C_k) &= \sum_{f=1}^N \left[x_f \ln p_{k,f} + (1- x_f) \ln (1-p_{k,f})\right]\\
&= \sum_{f=1}^N x_f \ln \frac{p_{k,f}}{1-p_{k,f}} + \sum_{f=1}^N \ln (1-p_{k,f})
\end{aligned}
$$

Finally, for obtaining **the discriminant function** of a class $C_k$, we have to also consider the prior probability:

$$d_k(\mathbf{x}) = \ln p(\mathbf{x}|C_k) + \ln P(C_k) = \sum_{f=1}^N x_f \underbrace{\ln \frac{p_{k,f}}{1-p_{k,f}}}_{w_{k,f}} + \underbrace{\ln P(C_k) + \sum_{f=1}^N \ln (1-p_{k,f})}_{b_k} = \mathbf{w}_k^T\mathbf{x} + b_k$$

As we can see, **it is a linear discriminant function**, like the ones in notebook 7.1!

**Implementation detail:** it is recommended to avoid values of 0 or 1 for $p_{k,f}$ since numerical problems may appear. Instead, take values close to 0 or 1. We will see this in a later assignment.

### **<span style="color:green"><b><i>ASSIGNMENT 1a: Setting up the data</i></b></span>**

The first step for training the classifier is preparing the data (this is often called the preprocessing step). In the context of our problem, this is done by vectorizing each image. 

**Our task** is to construct a matrix containing all the images' data, which can be found at: `./images/train_binary/imagen{0-9}_{1-500}.png`. This matrix will have 3 dimensions: $(\text{n}\_\text{pixels}, \text{n}\_\text{images}, \text{n}\_\text{classes})$. Thus, each column of the matrix (second dimension) would contain a vectorized version of its corresponding image, having `n_images` in total. The third dimension indexes the different classes, having this problem a total of 10 (digits from 0 to 9).

The image below illustrates the vectorization process of an image:

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/vectorization_dataset_matrix.png" width="1100"/>
    <figcaption>Fig. 2. Vectorization of a 28×28 binary image: its rows are stacked into a feature vector with 784 elements, which is stored as a column of the dataset matrix.</figcaption>
</center>$\\[5pt]$

In this way, **you have to** read all the images (there are 500 for each digit) and obtain the mentioned matrix storing all the data. Notice that the provided images are grayscale, so you will have to **binarize** them first using OpenCV, thus they can be used to build a Bernoulli distribution.

In [ ]:
## Training

# Initialize matrix
r,c = 28,28
dataset = np.zeros((r*c,500,10))

# For each class
for number in range(10):
    # For each image in the class
    for i in range(1,501):
        # Read the image
        path = images_path + "train_binary/imagen" + str(None) + "_" + str(i) + ".png"
        image = cv2.imread(path,0)
        # Binarize it
        _,binarized = cv2.threshold(None,100,1,cv2.THRESH_BINARY)
        # Vectorize it
        vector_binarized = binarized.reshape((-1,1))
        # Store it in the dataset matrix
        dataset[:,i-1,number] = vector_binarized[:,0]

### **<span style="color:green"><b><i>ASSIGNMENT 1b: Computing probabilities</i></b></span>**

After the preprocessing step, we will compute the different $p_{k,f}$ modeling the probability that $x_f = 1$ if $\mathbf{x}$ belongs to $C_k$. These probabilities will allow us to compute the weights in a later step. Recall how the discriminant functions are built:

$$d_k(\mathbf{x}) = \ln p(\mathbf{x}|C_k) + \ln P(C_k) = \sum_{f=1}^N x_f \underbrace{\ln \frac{p_{k,f}}{1-p_{k,f}}}_{w_{k,f}} + \underbrace{\ln P(C_k) + \sum_{f=1}^N \ln (1-p_{k,f})}_{b_k}$$

Luckily, considering a Bernoulli distribution, computing these probabilities is straightforward. We just have to count the number of times that each pixel (feature $x_f$) takes the value `1` in the images of each class, and divide it by the number of images of that class ($n_k$). Using the one-hot labels $y_{i,k}$ of the $n$ training images, as in notebook 7.2:

$$p_{k,f} = \frac{1}{n_k}\sum_{i=1}^{n} y_{i,k}\, x_{i,f}$$

being $x_{i,f}$ the $f$-th feature (pixel) of the $i$-th training image.

Regarding our problem:
- $n_k=500$ as we have $500$ images for each class.
- The index of features $f$ ranges from $1$ to $N=784$, as images have $28\times28$ pixels (in the code, NumPy indices go from $0$ to $783$).

The following image illustrates the process of computing these probabilities:

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/probabilities_estimation.png" width="1100"/>
    <figcaption>Fig. 3. Estimation of the probability of each pixel being 1 for each class: the mean of the images of the class, stored as a column of the probabilities matrix.</figcaption>
</center>$\\[5pt]$

**What to do?** 
1. Construct a 2D matrix **containing all the probabilities** (`probabilities[f,k]` $= p_{k,f}$). 
2. Then, we are going to visualize the **prototype images**, also called **heatmaps**. There is a heatmap for each digit. In these images, the value of a pixel represents its probability of being `1`, that is, of belonging to the stroke of the digit. For doing that, we have to **unvectorize the probabilities** in the matrix (2D $\rightarrow$ 3D) and **plot the heatmaps**. This is also a good way to check if we are doing well.

You should obtain something similar to this:

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/heatmaps.png" width="700"/>
    <figcaption>Fig. 4. Heatmaps of the probabilities of each pixel being 1 for each digit.</figcaption>
</center>$\\[5pt]$

*Tip: try to use [`np.sum()`](https://numpy.org/doc/stable/reference/generated/numpy.sum.html) for computing the probabilities in order to avoid loops, which are usually slower (check the `axis` argument).* ***You can compute probabilities in just one code line!***

In [ ]:
# Compute probabilities
probabilities = np.sum(None, axis=None)/None

# Reshape to obtain heatmaps
heatmaps = probabilities.reshape(r,c,10)

# Plot heatmaps
for i in range(2):
    for j in range(5):
        plt.subplot(2,5, i*5+(j+1))
        plt.title("Number " + str(i*5+j) + " probabilities")
        plt.imshow(heatmaps[:,:,i*5+j])

### **<span style="color:green"><b><i>ASSIGNMENT 1c: Computing weights</i></b></span>**

Once we have retrieved the probabilities $p_{k,f}$, we are ready to obtain **the weights** and **the bias** that define the discriminant function of each class. This is the last training step! Recall that they are defined as:

$$
\begin{aligned}
w_{k,f} &= \ln \frac{p_{k,f}}{1-p_{k,f}}\\[5pt]
b_k &= \ln P(C_k) + \sum_{f=1}^N \ln(1- p_{k,f})
\end{aligned}
$$

Being:
- $k$: one of the 10 possible classes, and
- $f$: a feature (pixel) in the range $1,\dots,784$.

**What to do?** Construct a 2D matrix containing the weights of the classifier, with the bias $b_k$ of each class in its last row, that is, the augmented weight vector $\tilde{\mathbf{w}}_k = [w_{k,1}, \ldots, w_{k,N}, b_k]^T$ of each class in a column (see notebook 7.1). Consider the following:

- Since we can't neither divide by 0 nor compute the logarithm of 0, during the weights computation, **0 and 1 probabilities should be replaced by close numbers** (e.g. 0.001 and 0.999). This can be done using [`np.where()`](https://numpy.org/doc/1.18/reference/generated/numpy.where.html).
- As we don't have any prior information about the occurrences of each class, the prior probability $P(C_k)$ should be $1/M = 0.1$ for each class $C_k$.

*Hint: For applying the natural logarithm to all the elements of a matrix, you can use [`np.log()`](https://numpy.org/doc/1.18/reference/generated/numpy.log.html?highlight=log#numpy.log).*

In [ ]:
## Weight calculation

# Replace 0 and 1 values
probabilities = np.where(probabilities==0, 0.001, probabilities)
probabilities = np.where(probabilities==1, 0.999, probabilities)

# Initialize matrix
weights = np.zeros((r*c+1,10))

# Compute weights
weights[:-1,:] = np.log(None/(1-None))
weights[-1,:] = np.log(0.1) + np.sum(np.log(1-None),axis=0)

### 7.3.2 Testing the classifier

In the last step we retrieved the weights $w_{k,f}$ and the biases $b_k$, so now we have all the building blocks needed to design the discriminant function of each class $d_k(\mathbf{x})$, $k=0,\ldots,9$. These discriminant functions permit us to evaluate a new vector $\mathbf{x}$ corresponding to a new vectorized image and retrieve its most probable class:

$$d_k(\mathbf{x}) = \sum_{f=1}^{N} w_{k,f}\, x_f + b_k = \tilde{\mathbf{w}}_k^T\tilde{\mathbf{x}} \quad k = 0,\ldots, 9$$

In this way, each discriminant function, after evaluating the vector of features, will return a number. As this is the **MAP discriminant** (with equal priors, a **maximum-likelihood classifier**), the assigned class will be the one whose discriminant function returns the highest value, $\hat{C}(\mathbf{x}) = \operatorname*{arg\,max}_k d_k(\mathbf{x})$.

### **<span style="color:green"><b><i>ASSIGNMENT 2</i></b></span>**

For this exercise, it is provided a NumPy matrix called `digitos.npy`, which is located at `./images/test_binary/`. It is a 1D matrix with 50 elements containing the ground truth information of 50 images, `./images/test_binary/timage{1-50}.png`. That is, the matrix codifies the digit, from 0 to 9, that each image contains. 

**Your task** is to classify those 50 images and check if the output of your classifier matches with the actual class, as provided in `digitos.npy`. In this way, we can estimate how good is our classifier.

*Note: If you trained your classifier with the provided images, you should expect 41/50 hits, or a 82% of accuracy.*

In [ ]:
## Testing

# Load provided matrix
results = np.load(images_path + "test_binary/digitos.npy")
hits = 0

# Classify each testing image
for i in range(1,51):
    f_pesos = np.zeros((10))
    # Read the image
    path = images_path + "test_binary/timage" + str(i) + ".png"
    image = cv2.imread(path,0)
    # Binarize it
    _,binarized = cv2.threshold(None,100,1,cv2.THRESH_BINARY)
    # Vectorize it
    vector_binarized = binarized.reshape((-1,1))
    vector_binarized = np.append(vector_binarized,[1])
    # Evaluate the discriminant functions
    for j in range(10):
        f_pesos[j] = np.sum(weights[:,j] * None)
    # Get the maximum
    res = np.where(f_pesos == np.amax(f_pesos))
    if res == results[i-1]:
        hits = hits + 1
        
# Print the accuracy
print("Accuracy=" + str(hits) + "/50 (" + str(100.0*hits/50.0) + "%)" )

### **<span style="color:green"><b><i>EXTRA: ASSIGNMENT 3</i></b></span>**

Do you dare to try your brand new classifier with your own images? Brilliant! For that:

1. Write a digit in a paper and make a photo with your phone. The following image shows an example (you can try with different numbers, of course!).$\\[5pt]$
<img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/testcamera.jpeg" width="300"/>$\\[5pt]$
2. Then, segment the region of interest (crop the digit, see the left image in the figure below). 
3. Resize the image to $28\times28$ pixels (this is the input image size of our classifier). You can use [`cv2.resize()`](https://docs.opencv.org/4.x/da/d54/group__imgproc__transform.html#ga47a974309e9102f5f08231edc7e7529d) for that. 
4. Finally, binarize (see the right image in the figure below) and vectorize the image. 
<img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/extra_example.png" width="700"/>
5. Follow the steps done in *assignment 2* to classify the image and show the results!

In [ ]:
## Testing with our own image

# Read image and convert it to grayscale
color_image = cv2.imread(images_path + 'testcamera.jpeg', -1)
color_image = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)

# Crop ROI
color_image = color_image[380:500,1390:1510,:]
image = cv2.cvtColor(color_image, cv2.COLOR_RGB2GRAY)

# Binarize it
image = cv2.resize(image,(r,c))
_,binarized = cv2.threshold(image,130,1,cv2.THRESH_BINARY_INV)

# Reshape the image
vector_binarized = binarized.reshape((-1,1))
vector_binarized = np.append(vector_binarized,[1])

# Evaluate the discriminant functions
for j in range(10):
    f_pesos[j] = np.sum(weights[:,j] * vector_binarized)
    
# Get the maximum
res = np.where(f_pesos == np.amax(f_pesos))

# Plot the results
plt.subplot(121)
plt.imshow(color_image)
plt.title("Original image")
plt.subplot(122)
plt.imshow(binarized,cmap="gray")
print("Image classified as " + str(res[0]))

## Conclusion

Excellent! You have learned how to:

- design and train a naïve Bayes classifier based on the Bernoulli distribution for binary images,
- test and obtain the accuracy of the classifier,
- recognize digits in the context of plate characters (notice that the handwritten digits dataset used in the assignments could be replaced by any other containing plate characters!).

## <font color="orange"><b><i>AI Appendix</i></b></font>

- **Model/tool used:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- **What I used AI for:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain how you used AI in this notebook: whether you followed the proposed AI interactions, adapted them, skipped some of them, or used AI for additional purposes such as debugging, clarifying concepts, improving prompts or checking your reasoning.</i></p>

- **Most and least useful proposed AI interactions:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By proposed AI interactions, we mean the AI prompts explicitly included in the notebook. Choose one that helped you the most and one that helped you the least. Briefly explain why.</i></p>

- **Most useful personal AI interaction:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By personal AI interaction, we mean any additional prompt or follow-up question that you wrote yourself, beyond the prompts proposed in the notebook. Paste or summarize one, if any, and explain why it helped.</i></p>

- **What I verified, adapted or decided myself:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain what you checked, corrected, adapted or decided before using the AI output in your final work.</i></p>